# 02 · Anotación manual

Tres formas de anotar, según si la persona puede o no ver a los modelos:

| Sección | Qué se ve | Dónde se guarda | Sirve para |
|---|---|---|---|
| **1. Asistida** (`AssistedApp`) | foto + valores de los 4 prompts y del triaje; se elige el correcto con un clic o se escribe otro | `annotations_assisted/HZ.jsonl` | ir rápido; entrenar |
| **2. Revisión con letra y recuadro** (`AnnotationApp(prefill='assisted')`) | formulario prellenado con lo de la sección 1 | `annotations/manual_HZ.jsonl`, con `prefilled_from='assisted'` | dejar la anotación completa (letra, recuadro, fotos que faltan) |
| **3. A ciegas** (`AnnotationApp`) | solo la foto, **sin modelos** | `annotations/manual_HZ.jsonl` | medir los prompts (set de validación) |

**Regla importante:** lo anotado en 1 y 2 **no es a ciegas** (se vieron los modelos). Sirve para entrenar, no para evaluar los prompts. Para evaluar se usa solo lo anotado en 3.

Reglas de anotación (detalle en la guía de Martin y en el contexto del proyecto):
- Una fila por foto e individuo; letras a, b, c… en sentido horario desde arriba a la izquierda, en la primera foto donde aparece cada hueso, y la misma letra en todas las fotos del registro.
- Lado, erosión y calidad por foto. Calidad: high = nada encima o un par de granos; medium = varios granos, pasto, algas, dedos… tapan parte (~5-60 %); low = gran parte tapada (> ~60 %) o borrosa.
- Erosión: 1 casi intacto; 2 faltan partes (< 10 %); 3 falta ≥ 10 % o es un fragmento.
- Cada *Save* agrega una línea; vale la última por registro + foto + letra. Nada se borra.

| Notebook | Para qué |
|---|---|
| `01_descarga_y_modelos` | bajar registros de iNaturalist y correr el triaje y los 4 prompts |
| `02_anotacion` | anotar: asistida (elegir el mejor valor de los modelos), revisión con letra y recuadro, o a ciegas |
| `03_revision_y_metricas` | revisión interna: ranking de prompts, moda, métricas, persona vs modelos, duplicados |
| `04_revision_con_martin` | lo que se le muestra a Martin: qué hicimos, resultados, fotos, preguntas y el Excel |
| `05_entrega_excel` | generar y revisar el Excel de entrega |

La versión oficial de cada notebook está en GitHub (`notebooks/`, sin resultados). Para trabajar: abrirlo desde GitHub en Colab, *Archivo → Guardar una copia en Drive* y moverla a `MyDrive/sepia-inaturalist/notebooks/` (con la fecha en el nombre).

In [ ]:
# --- Preparacion (correr siempre primero) ---
# Monta Drive, clona o actualiza el repo y carga el codigo recien bajado.
from google.colab import drive, output, userdata
drive.mount('/content/drive')
output.enable_custom_widget_manager()          # interfaces (ipywidgets) en Colab

import os, sys, importlib
if not os.path.exists('/content/sepia-inaturalist'):     # sesion nueva: clonar el repo
    token = userdata.get('GITHUB_TOKEN')
    !git clone -q https://{token}@github.com/hugozeballos/sepia-inaturalist.git /content/sepia-inaturalist
%cd /content/sepia-inaturalist
!git pull -q
!git log -1 --oneline
!pip install -q jupyter-bbox-widget          # recuadros en la interfaz de anotacion

sys.path.insert(0, '/content/sepia-inaturalist/src')
os.environ['PYTHONPATH'] = '/content/sepia-inaturalist/src'   # para los !python scripts/...
for m in [m for m in sys.modules if m == 'sepia' or m.startswith('sepia.')]:
    del sys.modules[m]                          # descartar la version vieja en memoria
sys.path_importer_cache.pop('/content/sepia-inaturalist/src', None)
importlib.invalidate_caches()
import sepia

# Los 4 prompts que se usan desde el 4 nov (detalle en docs/prompt_versions.md)
VERSIONS = ['photo_traits_v14', 'photo_traits_v13', 'photo_traits_v12', 'photo_traits_v8']
ANNOTATOR = 'HZ'

In [ ]:
from sepia.annotate import AnnotationApp
from sepia.assisted import AssistedApp, load_assisted

CONFIG = 'configs/sector_b.yaml'     # <- corrida a anotar (pilot_a1, a2_test, sector_a ... sector_e)
BONEBOX = 'bonebox_v1'              # recuadros del modelo (notebook 01, paso 5b)
cfg = sepia.load_config(CONFIG)
obs, pho = sepia.load_tables(cfg)
print(cfg.base_dir, f"· {len(obs)} registros, {len(pho)} fotos")

## 1. Anotación asistida: elegir el mejor valor de los modelos

Para cada atributo (conteo, lado, erosión, calidad), en orden de prioridad:
1. **my value**: un valor propio, elegido de una lista cerrada;
2. **clic** en el valor de un prompt (o del triaje, en el conteo);
3. si **todos coinciden** (también `unclear`), se propone ese valor.

- Desacuerdos entre prompts en **amarillo**; la elección en azul.
- Campos manuales con valores por defecto: estado (según el triaje), epibiontes no, certeza 1, picoteo según el lado (dorsal → no, ventral → yes).
- Nada se guarda hasta apretar *Save*. Solo aparecen fotos donde algún prompt o el triaje ve 1 hueso. Si el conteo final no es 1, se guarda solo el conteo y la foto queda para la sección 2.

In [ ]:
AssistedApp(cfg, VERSIONS, annotator=ANNOTATOR, model_boxes=BONEBOX).show()   # recuadro del modelo en rojo


In [ ]:
# Lo guardado en la asistida (una linea por foto, vale la ultima)
a = load_assisted(cfg, ANNOTATOR)
print(f"{len(a)} fotos guardadas en la asistida")
if len(a):
    print("Quedan para la seccion 2 (conteo distinto de 1):",
          int(a['needs_manual_annotation'].fillna(False).sum()))

## 2. Revisión con letra y recuadro (y fotos que faltan)

Recorre **todas** las fotos de la corrida en orden de registro, empezando por la primera sin anotación manual.
- En las fotos que pasaron por la asistida, el formulario parte con esos valores (aviso celeste): revisar, elegir la **letra**, dibujar el **recuadro** (*Draw box* y arrastrar) y guardar.
- Las fotos con varios huesos o sin asistida se anotan a mano.
- **Recuadro:** en fotos con 1 hueso aparece el recuadro del modelo (`bonebox_v1`) ya dibujado, con aviso amarillo. *Save* lo acepta; *Draw box* lo reemplaza por uno propio; ✕ lo borra. Se guarda `box_source` (`model`, `model_edited` o `manual`) y el recuadro propuesto (`model_box`). Con varios huesos, el recuadro se dibuja a mano.
- Lo guardado lleva la marca `prefilled_from='assisted'`.

In [ ]:
AnnotationApp(cfg, ANNOTATOR, prefill='assisted', model_boxes=BONEBOX).show()


## 2b. Agregar recuadros a fotos ya anotadas (A1, A2)

Las fotos anotadas antes de tener recuadros (A1 no tiene ninguno). Recorre solo las fotos con **un** individuo anotado, sin recuadro y con recuadro del modelo: mirar, *Save* (o ajustar) y siguiente. Los demás valores quedan como estaban. Antes, correr `bonebox` en esa corrida (notebook 01, paso 5b, con su `CONFIG`).

In [ ]:
from sepia.annotate import photos_without_box
todo = photos_without_box(cfg, BONEBOX)
print(f"{len(todo)} fotos con 1 individuo, sin recuadro y con recuadro del modelo")
if todo:
    AnnotationApp(cfg, ANNOTATOR, photo_ids=todo, model_boxes=BONEBOX).show()


## 3. Anotación a ciegas

Sin ver ningún resultado de los modelos. Es la que se usa para **medir** los prompts.

**Pendiente:** apartar un **set de validación fijo** de 50 a 80 fotos nuevas (que no se hayan mirado en ninguna interfaz con modelos) y anotarlo aquí. Sin ese set, las cifras de los prompts salen optimistas (ver notebook 03).

Sin recuadros del modelo (`model_boxes` queda en `None`).

In [ ]:
# Para anotar a ciegas una corrida (o una lista de registros: ids=[...])
AnnotationApp(cfg, ANNOTATOR).show()

## 4. Control del avance

In [ ]:
from sepia.annotate import load_events, done_photos, work_queue
ev = load_events(cfg)
done = done_photos(ev)
q = work_queue(cfg)
pending = q[[(int(o), int(p)) not in done for o, p in zip(q['observation_id'], q['photo_id'])]]
ind = ev['individual']
no_box = ind['crop_box'].isna().sum() if 'crop_box' in ind else len(ind)
assisted = (ind.get('prefilled_from') == 'assisted').sum() if 'prefilled_from' in ind else 0
print(f"Fotos anotadas: {len(q) - len(pending)}/{len(q)} · pendientes en registros: "
      f"{sorted(pending['observation_id'].unique().tolist())}")
print(f"Individuos: {len(ind)} · sin recuadro: {int(no_box)} · prellenados de la asistida: {int(assisted)}")